### Dataset and Task Metadata

In [82]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="lymphoma9",
    dataset_year="2017",
    domain_str="business & marketing",
    # Data Source
    dataset_source="GOV Website",
    original_dataset_source_download_link="https://llmpp.ccr.cancer.gov/lymphoma/data.shtml",
    download_description="""
Download from UCI and uzip data to a predefined folder.
mkdir -p local-data-warehouse/lymphoma9/ && wget -P local-data-warehouse/lymphoma9/ https://llmpp.ccr.cancer.gov/lymphoma/data/figure1/figure1.cdt 
""",
    # References
    academic_reference_bibtex="""@article{alizadeh2000lymphoma9,
  title={Distinct types of diffuse large B-cell lymphoma identified by gene expression profiling},
  author={Alizadeh, Ash A and Eisen, Michael B and Davis, R Eric and Ma, Chi and Lossos, Izidore S and Rosenwald, Andreas and Boldrick, Jennifer C and Sabet, Hajeer and Tran, Truc and Yu, Xin and others},
  journal={Nature},
  volume={403},
  number={6769},
  pages={503--511},
  year={2000},
  publisher={Nature Publishing Group UK London}
}
""",
    academic_reference_bibtex_key="alizadeh2000lymphoma9",
    license="Public Domain",
    data_tags=["IID"],
    curation_comments="""
- We transposed the dataset, so that every gene is a feature
- We dropped the AID column, which is just an identifier and not a relevant feature
- Anomaly: we can't find the "class" column that is used on OpenML as a target (https://www.openml.org/search?type=data&sort=runs&id=45095&status=active) in the original data mentioned in the cited paper (https://llmpp.ccr.cancer.gov/lymphoma/data/figure1/figure1.cdt). There is no valid target.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="",
)

## Preprocessing

In [83]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/figure1.cdt", sep="\t", encoding="latin-1", header=0)

df = df.set_index('GID')
df = df.T

df.drop(columns="AID", inplace=True)
feature_names = df.columns
cat_features = ["GENE1835X"]
num_features = [col for col in df.columns if col not in cat_features]

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df[cat_features] = df[cat_features].astype("category")
df[num_features] = df[num_features].apply(pd.to_numeric, errors='coerce').astype('float64')

In [84]:
# Use if needed to get see all cols of pandas dataframes
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
df.head()

GID GENE1835X  GENE1836X  GENE1865X  GENE1380X  GENE1933X  GENE1932X  \
0       -0.01       0.01       0.27       0.10       0.42       0.99   
1        0.42       0.03      -0.76       0.31       0.14      -0.10   
2       -0.03      -0.38       0.01       0.29      -0.11      -0.13   
3        0.38       0.21       0.33      -0.02      -0.11      -0.05   
4        0.34        NaN       0.36       0.11       0.15       0.19   

GID  GENE1931X  GENE1930X  GENE3129X  GENE3126X  GENE0X  GENE3115X  GENE3116X  \
0         0.62       0.19       0.39       0.42    0.00       0.11      -0.20   
1        -0.35      -0.36       0.10       0.60    0.28       0.03      -0.07   
2        -0.36       0.46       0.19      -0.05   -0.18       0.14      -0.56   
3        -0.37      -0.54      -0.18      -0.72   -0.13       0.25       0.00   
4        -0.11       0.27       0.59       0.55   -0.11       0.31      -0.25   

GID  GENE3117X  GENE3118X  GENE3073X  GENE3072X  GENE3067X  GENE3068X  \
0        -0.31      -0.29       0.17      -0.16       1.73       0.05   
1        -0.08      -0.42       0.80      -0.04      -1.16      -1.70   
2          NaN        NaN       0.00      -0.22      -0.89        NaN   
3        -0.34       0.35      -0.60       0.60      -0.08      -0.82   
4         1.04       0.90       0.15       0.68       0.13       0.90   

GID  GENE3069X  GENE2584X  GENE3070X  GENE1843X  GENE3166X  GENE3165X  \
0        -0.04      -0.32      -0.15      -0.56      -0.22      -0.66   
1        -0.97       0.16      -0.49      -0.66       0.21      -0.23   
2        -2.08       0.71       0.18       0.75       0.00        NaN   
3        -1.93      -0.35      -0.36      -0.49      -0.39      -0.49   
4         0.86        NaN       0.20       0.04       0.58        NaN   

GID  GENE1885X  GENE3385X  GENE3384X  GENE1110X  GENE1109X  GENE1108X  \
0         0.16      -0.51      -0.83       0.06       0.69       0.35   
1        -0.46      -0.41      -0.28      -0.53      -0.28       0.12   
2        -0.97      -1.25      -1.13      -1.53       1.15       0.45   
3         0.44       0.27       0.23       0.40       0.76       0.15   
4          NaN      -0.78      -0.48       0.61       0.19       0.60   

GID  GENE1107X  GENE4005X  GENE4006X  GENE4007X  GENE4008X  GENE4009X  \
0         0.80       1.66       2.08       2.13       1.38      -0.49   
1         0.14      -1.19      -0.34      -0.49      -0.28      -0.55   
2        -0.72      -0.42      -1.01      -0.74      -0.96       0.12   
3         0.33       0.29      -1.83      -2.18      -2.12      -1.85   
4          NaN       0.11      -0.08      -0.06       0.00       0.67   

GID  GENE4010X  GENE4011X  GENE4012X  GENE4013X  GENE4014X  GENE4015X  \
0        -0.01       0.47      -1.12        NaN      -0.05      -0.01   
1        -0.44      -0.62      -0.94      -1.21      -0.72      -0.14   
2          NaN        NaN      -1.24      -0.49      -0.98      -0.73   
3         0.20       0.45       1.01       1.16       1.51       1.25   
4          NaN        NaN        NaN        NaN        NaN        NaN   

GID  GENE4016X  GENE3884X  GENE3885X  GENE3886X  GENE3887X  GENE3888X  \
0        -1.33      -1.09      -0.16      -0.57      -0.28      -0.67   
1         0.14      -0.52      -1.31      -1.46      -0.97      -0.76   
2        -0.83      -0.35       0.58       0.50       0.38       0.50   
3        -0.22      -0.85      -1.37      -1.01      -1.26      -1.92   
4        -1.35       0.14       0.04       0.06        NaN        NaN   

GID  GENE3889X  GENE3890X  GENE1114X  GENE1115X  GENE3936X  GENE3937X  \
0        -0.36      -0.56      -0.28      -0.44       0.14       0.03   
1        -1.31      -0.40       0.15       0.34      -0.39      -1.07   
2        -0.22      -0.91       0.39       0.93      -0.01       0.27   
3        -0.44       0.55      -0.12       0.24      -0.57      -0.38   
4          NaN       0.85       0.10       0.16        NaN        NaN   

GID  GE

## Data Checks

In [85]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

ValueError: target_feature '' is not in the DataFrame.

In [73]:
# Sample Rows
df_head

GID GENE1835X  GENE1836X  GENE1865X  GENE1380X  GENE1933X  GENE1932X  \
0       -0.01       0.01       0.27       0.10       0.42       0.99   
1        0.42       0.03      -0.76       0.31       0.14      -0.10   
2       -0.03      -0.38       0.01       0.29      -0.11      -0.13   
3        0.38       0.21       0.33      -0.02      -0.11      -0.05   
4        0.34        NaN       0.36       0.11       0.15       0.19   

GID  GENE1931X  GENE1930X  GENE3129X  GENE3126X  GENE0X  GENE3115X  GENE3116X  \
0         0.62       0.19       0.39       0.42    0.00       0.11      -0.20   
1        -0.35      -0.36       0.10       0.60    0.28       0.03      -0.07   
2        -0.36       0.46       0.19      -0.05   -0.18       0.14      -0.56   
3        -0.37      -0.54      -0.18      -0.72   -0.13       0.25       0.00   
4        -0.11       0.27       0.59       0.55   -0.11       0.31      -0.25   

GID  GENE3117X  GENE3118X  GENE3073X  GENE3072X  GENE3067X  GENE3068X  \
0        -0.31      -0.29       0.17      -0.16       1.73       0.05   
1        -0.08      -0.42       0.80      -0.04      -1.16      -1.70   
2          NaN        NaN       0.00      -0.22      -0.89        NaN   
3        -0.34       0.35      -0.60       0.60      -0.08      -0.82   
4         1.04       0.90       0.15       0.68       0.13       0.90   

GID  GENE3069X  GENE2584X  GENE3070X  GENE1843X  GENE3166X  GENE3165X  \
0        -0.04      -0.32      -0.15      -0.56      -0.22      -0.66   
1        -0.97       0.16      -0.49      -0.66       0.21      -0.23   
2        -2.08       0.71       0.18       0.75       0.00        NaN   
3        -1.93      -0.35      -0.36      -0.49      -0.39      -0.49   
4         0.86        NaN       0.20       0.04       0.58        NaN   

GID  GENE1885X  GENE3385X  GENE3384X  GENE1110X  GENE1109X  GENE1108X  \
0         0.16      -0.51      -0.83       0.06       0.69       0.35   
1        -0.46      -0.41      -0.28      -0.53      -0.28       0.12   
2        -0.97      -1.25      -1.13      -1.53       1.15       0.45   
3         0.44       0.27       0.23       0.40       0.76       0.15   
4          NaN      -0.78      -0.48       0.61       0.19       0.60   

GID  GENE1107X  GENE4005X  GENE4006X  GENE4007X  GENE4008X  GENE4009X  \
0         0.80       1.66       2.08       2.13       1.38      -0.49   
1         0.14      -1.19      -0.34      -0.49      -0.28      -0.55   
2        -0.72      -0.42      -1.01      -0.74      -0.96       0.12   
3         0.33       0.29      -1.83      -2.18      -2.12      -1.85   
4          NaN       0.11      -0.08      -0.06       0.00       0.67   

GID  GENE4010X  GENE4011X  GENE4012X  GENE4013X  GENE4014X  GENE4015X  \
0        -0.01       0.47      -1.12        NaN      -0.05      -0.01   
1        -0.44      -0.62      -0.94      -1.21      -0.72      -0.14   
2          NaN        NaN      -1.24      -0.49      -0.98      -0.73   
3         0.20       0.45       1.01       1.16       1.51       1.25   
4          NaN        NaN        NaN        NaN        NaN        NaN   

GID  GENE4016X  GENE3884X  GENE3885X  GENE3886X  GENE3887X  GENE3888X  \
0        -1.33      -1.09      -0.16      -0.57      -0.28      -0.67   
1         0.14      -0.52      -1.31      -1.46      -0.97      -0.76   
2        -0.83      -0.35       0.58       0.50       0.38       0.50   
3        -0.22      -0.85      -1.37      -1.01      -1.26      -1.92   
4        -1.35       0.14       0.04       0.06        NaN        NaN   

GID  GENE3889X  GENE3890X  GENE1114X  GENE1115X  GENE3936X  GENE3937X  \
0        -0.36      -0.56      -0.28      -0.44       0.14       0.03   
1        -1.31      -0.40       0.15       0.34      -0.39      -1.07   
2        -0.22      -0.91       0.39       0.93      -0.01       0.27   
3        -0.44       0.55      -0.12       0.24      -0.57      -0.38   
4          NaN       0.85       0.10       0.16        NaN        NaN   

GID  GE

In [74]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,GENE1835X,category,3.0,3.03,74.0,"0.02, -0.13, 0.28, -0.01, -0.36, 0.04, 0.05, 0.06, 0.1, -0.78"
1,GENE3661X,float64,20.0,20.20,65.0,"-0.22, -0.54, 0.46, 1.03, -0.08, -0.32, 1.23, -0.25, 0.04, 0.24"
2,GENE3651X,float64,20.0,20.20,74.0,"0.3, -0.96, -1.9, -0.4, -0.34, -0.39, -2.65, -0.48, 1.27, -1.11"
3,GENE1903X,float64,20.0,20.20,64.0,"-0.03, 0.61, -0.16, 1.0, -0.25, -0.41, -0.04, -0.42, 0.98, -0.08"
4,GENE347X,float64,20.0,20.20,65.0,"0.12, 0.05, -0.26, -0.51, -0.09, -0.59, -0.7, -0.44, 1.0, -0.25"
5,GENE1386X,float64,20.0,20.20,70.0,"0.53, 0.0, 0.29, -0.08, -0.23, 0.52, -0.73, 0.25, -0.04, -1.4"
6,GENE1664X,float64,20.0,20.20,69.0,"1.0, -0.38, -0.2, 2.56, -1.2, 0.05, 1.13, 0.0, 2.0, 1.83"
7,GENE1810X,float64,20.0,20.20,75.0,"-0.12, 0.1, 0.27, 0.38, -0.64, -0.35, -0.65, -0.94, 2.07, -0.24"
8,GENE3538X,float64,19.0,19.19,61.0,"-0.25, 0.47, -0.05, 0.4, 0.56, -0.44, -0.46, 0.28, 0.25, 0.01"
9,GENE2562X,float64,19.0,19.19,64.0,"0.36, 0.16, -0.82, -0.78, 0.56, 0.0, 0.34, 0.43, -0.53, -0.19"


In [75]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
GENE1836X,82.0,193.106463,1749.242604,-1.49,15840.0
GENE1865X,96.0,155.586250,1524.398927,-1.80,14936.0
GENE1380X,91.0,163.160110,1556.281887,-0.80,14846.0
GENE1933X,86.0,173.935000,1612.853052,-1.15,14957.0
GENE1932X,96.0,142.379479,1394.468363,-1.17,13663.0
GENE1931X,94.0,145.474255,1410.573339,-1.20,13676.0
GENE1930X,98.0,139.444184,1380.066790,-0.89,13662.0
GENE3129X,98.0,177.871122,1761.408928,-1.15,17437.0
GENE3126X,97.0,176.453608,1737.558839,-1.62,17113.0
GENE0X,91.0,163.662857,1560.786305,-0.71,14889.0


In [76]:
# Categorical Feature Statistics
cat_stats

value  count   pct
column    rank                    
GENE1835X 1      0.02      4  4.04
          2     -0.13      3  3.03
          3      0.28      3  3.03
          4      <NA>      3  3.03
          5      0.25      2  2.02

In [77]:
# Target Distribution
target_df

,count,pct
GENE1835X,,
0.02,4,4.04
-0.13,3,3.03
0.28,3,3.03
NaN,3,3.03
0.25,2,2.02
0.06,2,2.02
-0.36,2,2.02
0.22,2,2.02
0.1,2,2.02


## Task Curation

In [78]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=20, n_splits=3, test_size=None


In [79]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


ValueError: Input contains NaN

## Export

In [80]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

NameError: name 'splits_mold' is not defined